In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Inference

## CNN mel-spectrogram model (Milestone-3)

In [2]:
# """
# Kaggle inference notebook for CNN mel-spectrogram model.
# Uploaded cnn_best.pth (cnn-baseline-model) as a Kaggle dataset, then imported it here.
# """

# import os
# import numpy as np
# import pandas as pd
# import librosa
# import torch
# import torch.nn as nn
# import torchaudio.transforms as T
# from tqdm import tqdm
# import warnings
# warnings.filterwarnings("ignore")

In [3]:
# # --- Paths ---
# MODEL_PATH = '/kaggle/input/datasets/rohanpuntambekar/milestone-trained-models/cnn_best.pth'
# COMP_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
# TEST_CSV = os.path.join(COMP_ROOT, 'test.csv')
# MASHUPS_DIR = os.path.join(COMP_ROOT, 'mashups')


In [4]:
# DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(f"Using device: {DEVICE}")

In [5]:
# # --- Load checkpoint ---
# checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
# GENRES = checkpoint['genres']
# config = checkpoint['config']
# SR = config['sr']
# DURATION = config['duration']
# N_FFT = config['n_fft']
# HOP_LENGTH = config['hop_length']
# N_MELS = config['n_mels']
# N_SAMPLES = int(SR * DURATION)
# IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

In [6]:
# print(f"Genres: {GENRES}")
# print(f"Config: SR={SR}, duration={DURATION}, n_mels={N_MELS}")

In [7]:

# # --- Model definition  ---
# class GenreCNN(nn.Module):
#     def __init__(self, n_classes=10):
#         super().__init__()
#         self.features = nn.Sequential(
#             nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
#             nn.BatchNorm2d(16),
#             nn.ReLU(),
#             nn.MaxPool2d(kernel_size=2, stride=2),

#             nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(kernel_size=2, stride=2),

#             nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.MaxPool2d(kernel_size=2, stride=2),

#             nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.MaxPool2d(kernel_size=2, stride=2),
#         )
#         self.classifier = nn.Sequential(
#             nn.AdaptiveAvgPool2d((1, 1)),
#             nn.Flatten(),
#             nn.Dropout(0.3),
#             nn.Linear(128, 64),
#             nn.ReLU(),
#             nn.Dropout(0.2),
#             nn.Linear(64, n_classes),
#         )

#     def forward(self, x):
#         x = self.features(x)
#         x = self.classifier(x)
#         return x

In [8]:
# model = GenreCNN(n_classes=len(GENRES)).to(DEVICE)
# model.load_state_dict(checkpoint['model_state_dict'])
# model.eval()
# print("Model loaded successfully.")

In [9]:
# # Mel spectrogram transform
# mel_transform = T.MelSpectrogram(
#     sample_rate=SR, n_fft=N_FFT,
#     hop_length=HOP_LENGTH, n_mels=N_MELS
# )
# db_transform = T.AmplitudeToDB()

In [10]:
# def predict_mashup(audio_path):
#     """Load mashup, convert to mel spectrogram, predict genre."""
#     y, _ = librosa.load(audio_path, sr=SR, duration=DURATION)

#     if len(y) < N_SAMPLES:
#         y = np.pad(y, (0, N_SAMPLES - len(y)))
#     else:
#         y = y[:N_SAMPLES]

#     # Peak normalize
#     peak = np.max(np.abs(y))
#     if peak > 0:
#         y = y / peak

#     waveform = torch.tensor(y, dtype=torch.float32).unsqueeze(0)  # (1, n_samples)
#     mel = mel_transform(waveform)
#     mel_db = db_transform(mel)
#     mel_db = mel_db.unsqueeze(0).to(DEVICE)  # (1, 1, n_mels, time)

#     with torch.no_grad():
#         output = model(mel_db)
#         pred_idx = output.argmax(dim=1).item()

#     return IDX_TO_GENRE[pred_idx]

In [11]:
# # --- Load test data ---
# test_df = pd.read_csv(TEST_CSV)
# print(f"Test samples: {len(test_df)}")
# print(test_df.head())

In [12]:
# test_df['filename'] = test_df['id'].apply(lambda x: f'song{int(x):04d}.wav')
# fname_col = 'filename'

In [13]:
# # --- Predict ---
# predictions = []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Predicting"):
#     mashup_file = os.path.join(MASHUPS_DIR, str(row[fname_col]))
#     try:
#         pred = predict_mashup(mashup_file)
#     except Exception as e:
#         print(f"Error on {row[fname_col]}: {e}")
#         pred = 'rock'
#     predictions.append(pred)

In [14]:
# # --- Submission ---
# submission = pd.DataFrame({
#     'id': test_df['id'],
#     'genre': predictions
# })

In [15]:
# submission.to_csv('submission.csv', index=False)
# print(f"\nSubmission saved. Shape: {submission.shape}")
# print(submission['genre'].value_counts())

## CRNN (Milestone-4) 

In [16]:
# # Inference — CRNN (milestone 4)
# # Uploaded crnn_best.pth as a Kaggle dataset in the dataset directory Milestone Trained Models.

# import os
# import numpy as np
# import pandas as pd
# import torch
# import torch.nn as nn
# import torchaudio
# import torchaudio.transforms as T
# from tqdm import tqdm

In [17]:
# # paths
# TEST_CSV   = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
# MASHUP_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
# MODEL_PATH = "/kaggle/input/datasets/rohanpuntambekar/milestone-trained-models/crnn_best.pth"

In [18]:
# # audio config — matches extract_and_save_features() in helper_functions.py

# SR = 22050
# N_MELS = 128
# N_FFT = 2048
# HOP_LENGTH = 512
# N_CROPS = 5

# GENRES       = ["blues", "classical", "country", "disco", "hiphop",
#                 "jazz", "metal", "pop", "reggae", "rock"]
# IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}
# DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Device: {DEVICE}")

# mel_transform    = T.MelSpectrogram(sample_rate=SR, n_fft=N_FFT,
#                                     hop_length=HOP_LENGTH, n_mels=N_MELS)
# amplitude_to_db  = T.AmplitudeToDB()

In [19]:
# # CRNN architecture — matches model_skeleton.py from milestone4 

# class CRNN(nn.Module):
#     def __init__(self, num_classes=10):
#         super().__init__()
#         self.cnn = nn.Sequential(
#             nn.Conv2d(1, 32, kernel_size=3, padding=1),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#             nn.Conv2d(32, 64, kernel_size=3, padding=1),
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#         )
#         self.lstm = nn.LSTM(
#             input_size=2048, hidden_size=64,
#             num_layers=1, bidirectional=True, batch_first=True
#         )
#         self.fc = nn.Linear(128, num_classes)

#     def forward(self, x):
#         x = self.cnn(x)
#         b, c, f, t = x.shape
#         x = x.permute(0, 3, 1, 2).reshape(b, t, c * f)
#         x, _ = self.lstm(x)
#         x, _ = torch.max(x, dim=1)
#         return self.fc(x)

In [20]:
# # helpers

# def load_audio(path):
#     """Load audio with torchaudio, resample to SR if needed."""
#     try:
#         waveform, sr = torchaudio.load(path)
#         if sr != SR:
#             waveform = T.Resample(sr, SR)(waveform)
#         return waveform
#     except Exception as e:
#         print(f"  Warning: could not load {path} — {e}")
#         return torch.zeros(2, SR * 30)


# def waveform_to_melspec(waveform):
#     """
#     Replicates extract_and_save_features() exactly:
#     MelSpectrogram -> AmplitudeToDB -> average stereo to mono.
#     """
#     mel  = mel_transform(waveform)
#     spec = amplitude_to_db(mel)
#     if spec.shape[0] > 1:
#         spec = spec.mean(dim=0, keepdim=True)
#     return spec


# def predict_with_tta(model, waveform, n_crops, device):
#     """
#     Extract N evenly spaced crops, run each through the model,
#     and return the averaged softmax probability vector.
#     """
#     total_samples = waveform.shape[1]
#     crop_samples = SR * 30
#     probs_sum = torch.zeros(len(GENRES), device=device)

#     if total_samples <= crop_samples:
#         pad = crop_samples - total_samples
#         waveform = torch.nn.functional.pad(waveform, (0, pad))
#         spec = waveform_to_melspec(waveform).unsqueeze(0).to(device)
#         with torch.no_grad():
#             probs_sum += torch.softmax(model(spec), dim=1).squeeze(0)
#         return probs_sum

#     starts = np.linspace(0, total_samples - crop_samples, n_crops, dtype=int)
#     for start in starts:
#         crop = waveform[:, start: start + crop_samples]
#         spec = waveform_to_melspec(crop).unsqueeze(0).to(device)
#         with torch.no_grad():
#             probs_sum += torch.softmax(model(spec), dim=1).squeeze(0)

#     return probs_sum / n_crops

In [21]:
# # load model

# model = CRNN(num_classes=10).to(DEVICE)
# model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
# model.eval()
# print("CRNN loaded.")

In [22]:
# # inference

# test_df = pd.read_csv(TEST_CSV)
# test_df["filename"] = test_df["id"].apply(lambda x: f"song{int(x):04d}.wav")

# rows = []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
#     fpath    = os.path.join(MASHUP_DIR, row["filename"])
#     waveform = load_audio(fpath)
#     probs    = predict_with_tta(model, waveform, N_CROPS, DEVICE)
#     pred     = IDX_TO_GENRE[probs.argmax().item()]
#     rows.append({"id": row["id"], "genre": pred})

# submission = pd.DataFrame(rows)
# submission.to_csv("submission.csv", index=False)
# print(submission["genre"].value_counts())

## AST (Milestone-5)

In [23]:
# import os
# import numpy as np
# import pandas as pd
# import librosa
# import torch
# from transformers import AutoFeatureExtractor, ASTForAudioClassification
# from tqdm import tqdm

In [24]:
# # Paths
# TEST_CSV   = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
# MASHUP_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
# MODEL_PATH = "/kaggle/input/datasets/rohanpuntambekar/milestone-trained-models/ast_best_10epochs.pth"

In [25]:
# # audio config — matches milestone 5 training 

# SR       = 16000
# DURATION = 10.0
# N_CROPS  = 5

# AST_MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"

# GENRES       = ["blues", "classical", "country", "disco", "hiphop",
#                 "jazz", "metal", "pop", "reggae", "rock"]
# IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}
# DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Device: {DEVICE}")

In [26]:
# # load AST model
# feature_extractor = AutoFeatureExtractor.from_pretrained(AST_MODEL_ID)

# model = ASTForAudioClassification.from_pretrained(
#     AST_MODEL_ID,
#     num_labels=10,
#     ignore_mismatched_sizes=True
# )
# for param in model.audio_spectrogram_transformer.parameters():
#     param.requires_grad = False

# model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
# model = model.to(DEVICE)
# model.eval()
# print(f"AST loaded from {os.path.basename(MODEL_PATH)}.")

In [27]:
# # helpers

# def load_audio(path, sr):
#     try:
#         y, _ = librosa.load(path, sr=sr, mono=True)
#         return y
#     except Exception as e:
#         print(f"  Warning: could not load {path} — {e}")
#         return np.zeros(int(sr * DURATION), dtype=np.float32)


# def predict_with_tta(model, feature_extractor, audio, sr, duration, n_crops, device):
#     crop_len  = int(sr * duration)
#     total_len = len(audio)
#     probs_sum = torch.zeros(len(GENRES), device=device)

#     if total_len <= crop_len:
#         crops = [np.pad(audio, (0, crop_len - total_len))]
#     else:
#         starts = np.linspace(0, total_len - crop_len, n_crops, dtype=int)
#         crops  = [audio[s: s + crop_len] for s in starts]

#     for crop in crops:
#         inputs      = feature_extractor(crop, sampling_rate=sr,
#                                         return_tensors="pt", padding=True)
#         input_values = inputs["input_values"].to(device)
#         with torch.no_grad():
#             logits = model(input_values).logits
#         probs_sum += torch.softmax(logits, dim=1).squeeze(0)

#     return probs_sum / len(crops)

In [28]:
# # inference

# test_df = pd.read_csv(TEST_CSV)
# test_df["filename"] = test_df["id"].apply(lambda x: f"song{int(x):04d}.wav")

# rows = []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
#     fpath = os.path.join(MASHUP_DIR, row["filename"])
#     audio = load_audio(fpath, SR)
#     probs = predict_with_tta(model, feature_extractor, audio, SR, DURATION,
#                              N_CROPS, DEVICE)
#     pred  = IDX_TO_GENRE[probs.argmax().item()]
#     rows.append({"id": row["id"], "genre": pred})

# submission = pd.DataFrame(rows)
# submission.to_csv("submission.csv", index=False)
# print(submission["genre"].value_counts())

## CRNN (src-dev) 

In [29]:
# # Necessary libraries
# import os
# import random
# import numpy as np
# import pandas as pd
# import librosa
# import torch
# import torch.nn as nn
# from tqdm import tqdm

# # paths

# TEST_CSV   = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
# MASHUP_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
# MODEL_PATH = "/kaggle/input/datasets/rohanpuntambekar/my-models/crnn_src_best.pth"


# # audio config — matches train_crnn

# SR         = 22050
# DURATION   = 10.0
# N_MELS     = 128
# N_FFT      = 2048
# HOP_LENGTH = 256
# N_CROPS    = 5

# GENRES       = ["blues", "classical", "country", "disco", "hiphop",
#                 "jazz", "metal", "pop", "reggae", "rock"]
# IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}
# DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Device: {DEVICE}")


# # CRNN architecture — matches train_crnn

# class CRNN(nn.Module):
#     def __init__(self, num_classes=10, dropout=0.3):
#         super().__init__()
#         self.cnn = nn.Sequential(
#             nn.Conv2d(1, 32, kernel_size=3, padding=1),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#             nn.Dropout2d(dropout),

#             nn.Conv2d(32, 64, kernel_size=3, padding=1),
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#             nn.Dropout2d(dropout),

#             nn.Conv2d(64, 128, kernel_size=3, padding=1),
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#             nn.Dropout2d(dropout),
#         )
#         self.lstm = nn.LSTM(
#             input_size=2048,
#             hidden_size=128,
#             num_layers=2,
#             bidirectional=True,
#             batch_first=True,
#             dropout=dropout
#         )
#         self.fc = nn.Linear(128 * 2, num_classes)

#     def forward(self, x):
#         x = self.cnn(x)
#         b, c, f, t = x.shape
#         x = x.permute(0, 3, 1, 2).reshape(b, t, c * f)
#         x, _ = self.lstm(x)
#         x, _ = torch.max(x, dim=1)
#         return self.fc(x)



# # helpers

# def load_audio(path, sr):
#     try:
#         y, _ = librosa.load(path, sr=sr, mono=True)
#         return y
#     except Exception as e:
#         print(f"  Warning: could not load {path} — {e}")
#         return np.zeros(int(sr * DURATION), dtype=np.float32)


# def audio_to_melspec(y, sr, n_mels, n_fft, hop_length):
#     S    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels,
#                                           n_fft=n_fft, hop_length=hop_length)
#     S_db = librosa.power_to_db(S, ref=np.max).astype(np.float32)
#     S_db = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-9)
#     return torch.tensor(S_db).unsqueeze(0)


# def predict_with_tta(model, audio, sr, n_mels, n_fft, hop_length,
#                      crop_len, n_crops, device):
#     total_len = len(audio)
#     probs_sum = torch.zeros(len(GENRES), device=device)

#     if total_len <= crop_len:
#         y    = np.pad(audio, (0, crop_len - total_len))
#         spec = audio_to_melspec(y, sr, n_mels, n_fft, hop_length).unsqueeze(0).to(device)
#         with torch.no_grad():
#             probs_sum += torch.softmax(model(spec), dim=1).squeeze(0)
#         return probs_sum

#     starts = np.linspace(0, total_len - crop_len, n_crops, dtype=int)
#     for start in starts:
#         y    = audio[start: start + crop_len]
#         spec = audio_to_melspec(y, sr, n_mels, n_fft, hop_length).unsqueeze(0).to(device)
#         with torch.no_grad():
#             probs_sum += torch.softmax(model(spec), dim=1).squeeze(0)

#     return probs_sum / n_crops



# # load model

# model = CRNN(num_classes=10).to(DEVICE)
# model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
# model.eval()
# print("CRNN loaded.")


# # inference


# test_df  = pd.read_csv(TEST_CSV)
# test_df["filename"] = test_df["id"].apply(lambda x: f"song{int(x):04d}.wav")
# crop_len  = int(SR * DURATION)

# rows = []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
#     fpath = os.path.join(MASHUP_DIR, row["filename"])
#     audio = load_audio(fpath, SR)
#     probs = predict_with_tta(model, audio, SR, N_MELS, N_FFT, HOP_LENGTH,
#                              crop_len, N_CROPS, DEVICE)
#     pred  = IDX_TO_GENRE[probs.argmax().item()]
#     rows.append({"id": row["id"], "genre": pred})

# submission = pd.DataFrame(rows)
# submission.to_csv("submission.csv", index=False)
# print(submission["genre"].value_counts())

## AST (src-dev) Phase 1

In [30]:
# # Necessary libraries
# import os
# import numpy as np
# import pandas as pd
# import librosa
# import torch
# from transformers import AutoFeatureExtractor, ASTForAudioClassification
# from tqdm import tqdm

# # Paths
# TEST_CSV   = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
# MASHUP_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
# MODEL_PATH = "/kaggle/input/datasets/rohanpuntambekar/my-models/ast_src_best.pth"

# SR       = 16000
# DURATION = 10.0
# N_CROPS  = 5

# AST_MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"

# GENRES       = ["blues", "classical", "country", "disco", "hiphop",
#                 "jazz", "metal", "pop", "reggae", "rock"]
# IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}
# DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Device: {DEVICE}")


# feature_extractor = AutoFeatureExtractor.from_pretrained(AST_MODEL_ID)

# model = ASTForAudioClassification.from_pretrained(
#     AST_MODEL_ID,
#     num_labels=10,
#     ignore_mismatched_sizes=True
# )
# for param in model.audio_spectrogram_transformer.parameters():
#     param.requires_grad = False

# model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
# model = model.to(DEVICE)
# model.eval()
# print(f"AST loaded from {os.path.basename(MODEL_PATH)}.")


# def load_audio(path, sr):
#     try:
#         y, _ = librosa.load(path, sr=sr, mono=True)
#         return y
#     except Exception as e:
#         print(f"  Warning: could not load {path} — {e}")
#         return np.zeros(int(sr * DURATION), dtype=np.float32)


# def predict_with_tta(model, feature_extractor, audio, sr, duration, n_crops, device):
#     crop_len  = int(sr * duration)
#     total_len = len(audio)
#     probs_sum = torch.zeros(len(GENRES), device=device)

#     if total_len <= crop_len:
#         crops = [np.pad(audio, (0, crop_len - total_len))]
#     else:
#         starts = np.linspace(0, total_len - crop_len, n_crops, dtype=int)
#         crops  = [audio[s: s + crop_len] for s in starts]

#     for crop in crops:
#         inputs       = feature_extractor(crop, sampling_rate=sr,
#                                          return_tensors="pt", padding=True)
#         input_values = inputs["input_values"].to(device)
#         with torch.no_grad():
#             logits = model(input_values).logits
#         probs_sum += torch.softmax(logits, dim=1).squeeze(0)

#     return probs_sum / len(crops)


# test_df = pd.read_csv(TEST_CSV)
# test_df["filename"] = test_df["id"].apply(lambda x: f"song{int(x):04d}.wav")

# rows = []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
#     fpath = os.path.join(MASHUP_DIR, row["filename"])
#     audio = load_audio(fpath, SR)
#     probs = predict_with_tta(model, feature_extractor, audio, SR, DURATION,
#                              N_CROPS, DEVICE)
#     pred  = IDX_TO_GENRE[probs.argmax().item()]
#     rows.append({"id": row["id"], "genre": pred})

# submission = pd.DataFrame(rows)
# submission.to_csv("submission.csv", index=False)
# print(submission["genre"].value_counts())

## AST (src-dev) Phase 2

In [31]:
# # Necessary libraries
# import os
# import numpy as np
# import pandas as pd
# import librosa
# import torch
# from transformers import AutoFeatureExtractor, ASTForAudioClassification
# from tqdm import tqdm

# # Paths
# TEST_CSV   = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
# MASHUP_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
# MODEL_PATH = "/kaggle/input/datasets/rohanpuntambekar/my-models/ast_src_phase2_best.pth.pth"

# SR       = 16000
# DURATION = 10.0
# N_CROPS  = 5

# AST_MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"

# GENRES       = ["blues", "classical", "country", "disco", "hiphop",
#                 "jazz", "metal", "pop", "reggae", "rock"]
# IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}
# DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Device: {DEVICE}")


# feature_extractor = AutoFeatureExtractor.from_pretrained(AST_MODEL_ID)

# model = ASTForAudioClassification.from_pretrained(
#     AST_MODEL_ID,
#     num_labels=10,
#     ignore_mismatched_sizes=True
# )
# for param in model.audio_spectrogram_transformer.parameters():
#     param.requires_grad = False

# model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
# model = model.to(DEVICE)
# model.eval()
# print(f"AST loaded from {os.path.basename(MODEL_PATH)}.")


# def load_audio(path, sr):
#     try:
#         y, _ = librosa.load(path, sr=sr, mono=True)
#         return y
#     except Exception as e:
#         print(f"  Warning: could not load {path} — {e}")
#         return np.zeros(int(sr * DURATION), dtype=np.float32)


# def predict_with_tta(model, feature_extractor, audio, sr, duration, n_crops, device):
#     crop_len  = int(sr * duration)
#     total_len = len(audio)
#     probs_sum = torch.zeros(len(GENRES), device=device)

#     if total_len <= crop_len:
#         crops = [np.pad(audio, (0, crop_len - total_len))]
#     else:
#         starts = np.linspace(0, total_len - crop_len, n_crops, dtype=int)
#         crops  = [audio[s: s + crop_len] for s in starts]

#     for crop in crops:
#         inputs       = feature_extractor(crop, sampling_rate=sr,
#                                          return_tensors="pt", padding=True)
#         input_values = inputs["input_values"].to(device)
#         with torch.no_grad():
#             logits = model(input_values).logits
#         probs_sum += torch.softmax(logits, dim=1).squeeze(0)

#     return probs_sum / len(crops)


# test_df = pd.read_csv(TEST_CSV)
# test_df["filename"] = test_df["id"].apply(lambda x: f"song{int(x):04d}.wav")

# rows = []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
#     fpath = os.path.join(MASHUP_DIR, row["filename"])
#     audio = load_audio(fpath, SR)
#     probs = predict_with_tta(model, feature_extractor, audio, SR, DURATION,
#                              N_CROPS, DEVICE)
#     pred  = IDX_TO_GENRE[probs.argmax().item()]
#     rows.append({"id": row["id"], "genre": pred})

# submission = pd.DataFrame(rows)
# submission.to_csv("submission.csv", index=False)
# print(submission["genre"].value_counts())

## Ensemble of CRNN (src-dev) and AST (src-dev)

In [32]:
# import os
# import numpy as np
# import pandas as pd
# import librosa
# import torch
# import torch.nn as nn
# from transformers import AutoFeatureExtractor, ASTForAudioClassification
# from tqdm import tqdm

# TEST_CSV    = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
# MASHUP_DIR  = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
# AST_PATH    = "/kaggle/input/datasets/rohanpuntambekar/my-models/ast_src_phase2_best.pth.pth"
# CRNN_PATH   = "/kaggle/input/datasets/rohanpuntambekar/my-models/crnn_src_best.pth"

# # weight assigned to each model's softmax probabilities
# AST_WEIGHT  = 0.8
# CRNN_WEIGHT = 0.2

# SR_AST = 16000
# SR_CRNN = 22050
# DURATION = 10.0
# N_MELS = 128
# N_FFT  = 2048
# HOP_LENGTH = 256
# N_CROPS = 5

# AST_MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"

# GENRES       = ["blues", "classical", "country", "disco", "hiphop",
#                 "jazz", "metal", "pop", "reggae", "rock"]
# IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}
# DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Device: {DEVICE}")


# class CRNN(nn.Module):
#     def __init__(self, num_classes=10, dropout=0.3):
#         super().__init__()
#         self.cnn = nn.Sequential(
#             nn.Conv2d(1, 32, kernel_size=3, padding=1),
#             nn.BatchNorm2d(32),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#             nn.Dropout2d(dropout),
#             nn.Conv2d(32, 64, kernel_size=3, padding=1),
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#             nn.Dropout2d(dropout),
#             nn.Conv2d(64, 128, kernel_size=3, padding=1),
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.MaxPool2d(2),
#             nn.Dropout2d(dropout),
#         )
#         self.lstm = nn.LSTM(
#             input_size=2048, hidden_size=128,
#             num_layers=2, bidirectional=True,
#             batch_first=True, dropout=dropout
#         )
#         self.fc = nn.Linear(128 * 2, num_classes)

#     def forward(self, x):
#         x = self.cnn(x)
#         b, c, f, t = x.shape
#         x = x.permute(0, 3, 1, 2).reshape(b, t, c * f)
#         x, _ = self.lstm(x)
#         x, _ = torch.max(x, dim=1)
#         return self.fc(x)

# feature_extractor = AutoFeatureExtractor.from_pretrained(AST_MODEL_ID)

# ast_model = ASTForAudioClassification.from_pretrained(
#     AST_MODEL_ID, num_labels=10, ignore_mismatched_sizes=True
# )
# for param in ast_model.audio_spectrogram_transformer.parameters():
#     param.requires_grad = False
# ast_model.load_state_dict(torch.load(AST_PATH, map_location=DEVICE, weights_only=True))
# ast_model = ast_model.to(DEVICE)
# ast_model.eval()
# print("AST loaded.")

# crnn_model = CRNN(num_classes=10).to(DEVICE)
# crnn_model.load_state_dict(torch.load(CRNN_PATH, map_location=DEVICE, weights_only=True))
# crnn_model.eval()
# print("CRNN loaded.")


# def load_audio(path, sr):
#     try:
#         y, _ = librosa.load(path, sr=sr, mono=True)
#         return y
#     except Exception as e:
#         print(f"  Warning: could not load {path} — {e}")
#         return np.zeros(int(sr * DURATION), dtype=np.float32)


# def ast_tta(model, feature_extractor, audio, sr, duration, n_crops, device):
#     crop_len  = int(sr * duration)
#     total_len = len(audio)
#     probs_sum = torch.zeros(len(GENRES), device=device)

#     if total_len <= crop_len:
#         crops = [np.pad(audio, (0, crop_len - total_len))]
#     else:
#         starts = np.linspace(0, total_len - crop_len, n_crops, dtype=int)
#         crops  = [audio[s: s + crop_len] for s in starts]

#     for crop in crops:
#         inputs       = feature_extractor(crop, sampling_rate=sr,
#                                          return_tensors="pt", padding=True)
#         input_values = inputs["input_values"].to(device)
#         with torch.no_grad():
#             logits = model(input_values).logits
#         probs_sum += torch.softmax(logits, dim=1).squeeze(0)

#     return probs_sum / len(crops)


# def crnn_tta(model, audio, sr, n_mels, n_fft, hop_length,
#              crop_len, n_crops, device):
#     total_len = len(audio)
#     probs_sum = torch.zeros(len(GENRES), device=device)

#     def to_spec(y):
#         S    = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels,
#                                               n_fft=n_fft, hop_length=hop_length)
#         S_db = librosa.power_to_db(S, ref=np.max).astype(np.float32)
#         S_db = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-9)
#         return torch.tensor(S_db).unsqueeze(0)

#     if total_len <= crop_len:
#         y    = np.pad(audio, (0, crop_len - total_len))
#         spec = to_spec(y).unsqueeze(0).to(device)
#         with torch.no_grad():
#             probs_sum += torch.softmax(model(spec), dim=1).squeeze(0)
#         return probs_sum

#     starts = np.linspace(0, total_len - crop_len, n_crops, dtype=int)
#     for start in starts:
#         spec = to_spec(audio[start: start + crop_len]).unsqueeze(0).to(device)
#         with torch.no_grad():
#             probs_sum += torch.softmax(model(spec), dim=1).squeeze(0)

#     return probs_sum / n_crops


# test_df  = pd.read_csv(TEST_CSV)
# test_df["filename"] = test_df["id"].apply(lambda x: f"song{int(x):04d}.wav")
# crnn_crop_len = int(SR_CRNN * DURATION)

# rows = []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
#     fpath = os.path.join(MASHUP_DIR, row["filename"])

#     ast_audio  = load_audio(fpath, SR_AST)
#     crnn_audio = load_audio(fpath, SR_CRNN)

#     ast_probs  = ast_tta(ast_model, feature_extractor, ast_audio,
#                          SR_AST, DURATION, N_CROPS, DEVICE)
#     crnn_probs = crnn_tta(crnn_model, crnn_audio, SR_CRNN, N_MELS,
#                           N_FFT, HOP_LENGTH, crnn_crop_len, N_CROPS, DEVICE)

#     combined = AST_WEIGHT * ast_probs + CRNN_WEIGHT * crnn_probs
#     pred     = IDX_TO_GENRE[combined.argmax().item()]
#     rows.append({"id": row["id"], "genre": pred})

# submission = pd.DataFrame(rows)
# submission.to_csv("submission.csv", index=False)
# print(submission["genre"].value_counts())

## Ensemble of AST Phase 3 and MERT Phase 2 (src-dev)

In [33]:
# import os
# import numpy as np
# import pandas as pd
# import librosa
# import torch
# import torch.nn as nn
# from transformers import AutoFeatureExtractor, ASTForAudioClassification, Wav2Vec2FeatureExtractor, AutoModel
# from tqdm import tqdm

# TEST_CSV   = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
# MASHUP_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"
# AST_PATH   = "/kaggle/input/datasets/rohanpuntambekar/my-models/ast_src_phase3_best.pth"
# MERT_PATH  = "/kaggle/input/datasets/rohanpuntambekar/my-models/mert_src_phase2_best.pth"


# AST_WEIGHT  = 0.7
# MERT_WEIGHT = 0.3

# SR_AST  = 16000
# SR_MERT = 24000
# DURATION = 10.0
# N_CROPS  = 5

# AST_MODEL_ID  = "MIT/ast-finetuned-audioset-10-10-0.4593"
# MERT_MODEL_ID = "m-a-p/MERT-v1-95M"

# GENRES       = ["blues", "classical", "country", "disco", "hiphop",
#                 "jazz", "metal", "pop", "reggae", "rock"]
# IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}
# DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Device: {DEVICE}")


# class MERTClassifier(nn.Module):
#     def __init__(self, backbone, hidden_size=768, num_classes=10):
#         super().__init__()
#         self.backbone   = backbone
#         self.classifier = nn.Linear(hidden_size, num_classes)

#     def forward(self, input_values, attention_mask=None):
#         outputs = self.backbone(input_values=input_values,
#                                 attention_mask=attention_mask,
#                                 output_hidden_states=False)
#         pooled  = outputs.last_hidden_state.mean(dim=1)
#         return self.classifier(pooled)



# ast_feature_extractor  = AutoFeatureExtractor.from_pretrained(AST_MODEL_ID)
# mert_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
#     MERT_MODEL_ID, trust_remote_code=True
# )

# ast_model = ASTForAudioClassification.from_pretrained(
#     AST_MODEL_ID, num_labels=10, ignore_mismatched_sizes=True
# )
# for param in ast_model.audio_spectrogram_transformer.parameters():
#     param.requires_grad = False
# ast_model.load_state_dict(torch.load(AST_PATH, map_location=DEVICE, weights_only=True))
# ast_model = ast_model.to(DEVICE)
# ast_model.eval()
# print("AST loaded.")

# mert_backbone = AutoModel.from_pretrained(MERT_MODEL_ID, trust_remote_code=True)
# mert_model    = MERTClassifier(mert_backbone, hidden_size=768, num_classes=10)
# mert_model.load_state_dict(torch.load(MERT_PATH, map_location=DEVICE, weights_only=False))
# mert_model = mert_model.to(DEVICE)
# mert_model.eval()
# print("MERT loaded.")


# def load_audio(path, sr):
#     try:
#         y, _ = librosa.load(path, sr=sr, mono=True)
#         return y
#     except Exception as e:
#         print(f"  Warning: could not load {path} — {e}")
#         return np.zeros(int(sr * DURATION), dtype=np.float32)


# def ast_tta(model, feature_extractor, audio, sr, duration, n_crops, device):
#     crop_len  = int(sr * duration)
#     total_len = len(audio)
#     probs_sum = torch.zeros(len(GENRES), device=device)

#     if total_len <= crop_len:
#         crops = [np.pad(audio, (0, crop_len - total_len))]
#     else:
#         starts = np.linspace(0, total_len - crop_len, n_crops, dtype=int)
#         crops  = [audio[s: s + crop_len] for s in starts]

#     for crop in crops:
#         inputs       = feature_extractor(crop, sampling_rate=sr,
#                                          return_tensors="pt", padding=True)
#         input_values = inputs["input_values"].to(device)
#         with torch.no_grad():
#             logits = model(input_values).logits
#         probs_sum += torch.softmax(logits, dim=1).squeeze(0)

#     return probs_sum / len(crops)


# def mert_tta(model, feature_extractor, audio, sr, duration, n_crops, device):
#     crop_len  = int(sr * duration)
#     total_len = len(audio)
#     probs_sum = torch.zeros(len(GENRES), device=device)

#     if total_len <= crop_len:
#         crops = [np.pad(audio, (0, crop_len - total_len))]
#     else:
#         starts = np.linspace(0, total_len - crop_len, n_crops, dtype=int)
#         crops  = [audio[s: s + crop_len] for s in starts]

#     for crop in crops:
#         inputs       = feature_extractor(crop, sampling_rate=sr,
#                                          return_tensors="pt", padding=True)
#         input_values = inputs["input_values"].to(device)
#         with torch.no_grad():
#             logits = model(input_values)
#         probs_sum += torch.softmax(logits, dim=1).squeeze(0)

#     return probs_sum / len(crops)


# test_df             = pd.read_csv(TEST_CSV)
# test_df["filename"] = test_df["id"].apply(lambda x: f"song{int(x):04d}.wav")

# rows = []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
#     fpath = os.path.join(MASHUP_DIR, row["filename"])

#     ast_audio  = load_audio(fpath, SR_AST)
#     mert_audio = load_audio(fpath, SR_MERT)

#     ast_probs  = ast_tta(ast_model, ast_feature_extractor, ast_audio,
#                          SR_AST, DURATION, N_CROPS, DEVICE)
#     mert_probs = mert_tta(mert_model, mert_feature_extractor, mert_audio,
#                           SR_MERT, DURATION, N_CROPS, DEVICE)

#     combined = AST_WEIGHT * ast_probs + MERT_WEIGHT * mert_probs
#     pred     = IDX_TO_GENRE[combined.argmax().item()]
#     rows.append({"id": row["id"], "genre": pred})

# submission = pd.DataFrame(rows)
# submission.to_csv("submission.csv", index=False)
# print(submission["genre"].value_counts())

## AST (src-dev) Full Fine-tuned Model

In [34]:


# import os
# import numpy as np
# import pandas as pd
# import torch
# import librosa
# from transformers import AutoFeatureExtractor, ASTForAudioClassification
# from tqdm import tqdm


# MODEL_PATH = "/kaggle/input/datasets/rohanpuntambekar/my-models/ast_src_full_finetune_best.pth"

# TEST_CSV   = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
# MASHUP_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"

# AST_MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"
# SR           = 16000
# DURATION     = 10.0
# N_CROPS      = 5

# GENRES       = ["blues", "classical", "country", "disco", "hiphop",
#                 "jazz", "metal", "pop", "reggae", "rock"]
# IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Device: {DEVICE}")

# feature_extractor = AutoFeatureExtractor.from_pretrained(AST_MODEL_ID)

# model = ASTForAudioClassification.from_pretrained(
#     AST_MODEL_ID, num_labels=10, ignore_mismatched_sizes=True
# )
# for param in model.audio_spectrogram_transformer.parameters():
#     param.requires_grad = False

# model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
# model = model.to(DEVICE)
# model.eval()
# print(f"AST loaded from {os.path.basename(MODEL_PATH)}.")


# def load_audio(path, sr):
#     try:
#         y, _ = librosa.load(path, sr=sr, mono=True)
#         return y
#     except Exception as e:
#         print(f"  Warning: could not load {path} — {e}")
#         return np.zeros(int(sr * DURATION), dtype=np.float32)


# def predict_with_tta(model, feature_extractor, audio, sr, duration, n_crops, device):
#     crop_len  = int(sr * duration)
#     total_len = len(audio)
#     probs_sum = torch.zeros(len(GENRES), device=device)

#     if total_len <= crop_len:
#         crops = [np.pad(audio, (0, crop_len - total_len))]
#     else:
#         starts = np.linspace(0, total_len - crop_len, n_crops, dtype=int)
#         crops  = [audio[s: s + crop_len] for s in starts]

#     for crop in crops:
#         inputs       = feature_extractor(crop, sampling_rate=sr,
#                                          return_tensors="pt", padding=True)
#         input_values = inputs["input_values"].to(device)
#         with torch.no_grad():
#             logits = model(input_values).logits
#         probs_sum += torch.softmax(logits, dim=1).squeeze(0)

#     return probs_sum / len(crops)


# test_df = pd.read_csv(TEST_CSV)
# test_df["filename"] = test_df["id"].apply(lambda x: f"song{int(x):04d}.wav")

# rows = []
# for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
#     fpath = os.path.join(MASHUP_DIR, row["filename"])
#     audio = load_audio(fpath, SR)
#     probs = predict_with_tta(model, feature_extractor, audio, SR, DURATION,
#                              N_CROPS, DEVICE)
#     pred  = IDX_TO_GENRE[probs.argmax().item()]
#     rows.append({"id": row["id"], "genre": pred})

# submission = pd.DataFrame(rows)
# submission.to_csv("submission.csv", index=False)
# print(submission["genre"].value_counts())

## AST (src-dev) Full fine-tuned Model on 400 samples

In [35]:
import os
import numpy as np
import pandas as pd
import torch
import librosa
from transformers import AutoFeatureExtractor, ASTForAudioClassification
from tqdm import tqdm


MODEL_PATH = "/kaggle/input/datasets/rohanpuntambekar/my-models/ast_src_full_finetune_400_best.pth"

TEST_CSV   = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv"
MASHUP_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"

AST_MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"
SR           = 16000
DURATION     = 10.0
N_CROPS      = 5

GENRES       = ["blues", "classical", "country", "disco", "hiphop",
                "jazz", "metal", "pop", "reggae", "rock"]
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

feature_extractor = AutoFeatureExtractor.from_pretrained(AST_MODEL_ID)

model = ASTForAudioClassification.from_pretrained(
    AST_MODEL_ID, num_labels=10, ignore_mismatched_sizes=True
)
for param in model.audio_spectrogram_transformer.parameters():
    param.requires_grad = False

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model = model.to(DEVICE)
model.eval()
print(f"AST loaded from {os.path.basename(MODEL_PATH)}.")


def load_audio(path, sr):
    try:
        y, _ = librosa.load(path, sr=sr, mono=True)
        return y
    except Exception as e:
        print(f"  Warning: could not load {path} — {e}")
        return np.zeros(int(sr * DURATION), dtype=np.float32)


def predict_with_tta(model, feature_extractor, audio, sr, duration, n_crops, device):
    crop_len  = int(sr * duration)
    total_len = len(audio)
    probs_sum = torch.zeros(len(GENRES), device=device)

    if total_len <= crop_len:
        crops = [np.pad(audio, (0, crop_len - total_len))]
    else:
        starts = np.linspace(0, total_len - crop_len, n_crops, dtype=int)
        crops  = [audio[s: s + crop_len] for s in starts]

    for crop in crops:
        inputs       = feature_extractor(crop, sampling_rate=sr,
                                         return_tensors="pt", padding=True)
        input_values = inputs["input_values"].to(device)
        with torch.no_grad():
            logits = model(input_values).logits
        probs_sum += torch.softmax(logits, dim=1).squeeze(0)

    return probs_sum / len(crops)


test_df = pd.read_csv(TEST_CSV)
test_df["filename"] = test_df["id"].apply(lambda x: f"song{int(x):04d}.wav")

rows = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    fpath = os.path.join(MASHUP_DIR, row["filename"])
    audio = load_audio(fpath, SR)
    probs = predict_with_tta(model, feature_extractor, audio, SR, DURATION,
                             N_CROPS, DEVICE)
    pred  = IDX_TO_GENRE[probs.argmax().item()]
    rows.append({"id": row["id"], "genre": pred})

submission = pd.DataFrame(rows)
submission.to_csv("submission.csv", index=False)
print(submission["genre"].value_counts())

Device: cuda


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


AST loaded from ast_src_full_finetune_400_best.pth.


Inference: 100%|██████████| 3020/3020 [23:52<00:00,  2.11it/s]

genre
rock         352
blues        341
hiphop       319
reggae       319
pop          307
metal        306
jazz         304
disco        303
classical    244
country      225
Name: count, dtype: int64
